# Prepare FineWeb-Edu on your Mac

Open this notebook in the project with your local Python 3 kernel and choose **Run All**.
Your CPU streams FineWeb-Edu, tokenizes with Qwen3-0.6B-Base, and saves the corpus in
**Downloads → fineweb_edu**. Allow about 48 GB for tokens plus room for caches.
After the validation documents, the progress bar shows training tokens per second and time remaining.
Keep the Mac awake while this runs. Interrupting tokenization discards the partial run.


In [1]:
from pathlib import Path
import shutil
import sys

# Change this path if you want to use an external disk.
OUTPUT = Path.home() / "Downloads" / "fineweb_edu"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUTPUT}")
print(f"Free disk space: {shutil.disk_usage(OUTPUT.parent).free / 1e9:.1f} GB")

cwd = Path.cwd()
repo = next((p for p in (cwd, *cwd.parents) if (p / "src/base_model_research/data.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook inside the base-model-research project folder.")
sys.path.insert(0, str(repo / "src"))
from base_model_research.config import DEFAULT_TRAINING_TARGETS
from base_model_research.data import TokenCorpus, prepare_data
from tqdm.auto import tqdm


Output folder: /Users/seungjunlee/Downloads/fineweb_edu
Free disk space: 92.4 GB


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if not (OUTPUT / "manifest.json").is_file():
    print("Opening FineWeb-Edu. The first 2,048 documents go to validation.", flush=True)
    with tqdm(
        total=DEFAULT_TRAINING_TARGETS + 1, desc="Training tokens", unit="tok", unit_scale=True, mininterval=1
    ) as progress:
        prepare_data(OUTPUT, on_progress=progress.update)

train = TokenCorpus(OUTPUT, "train")
validation = TokenCorpus(OUTPUT, "validation")
manifest = train.manifest
if (
    train.total_tokens != DEFAULT_TRAINING_TARGETS + 1
    or manifest.get("dataset") != "HuggingFaceFW/fineweb-edu"
    or manifest.get("dataset_config") != "sample-100BT"
    or manifest.get("tokenizer") != "Qwen/Qwen3-0.6B-Base"
    or manifest.get("validation_documents") != 2048
    or not (OUTPUT / "tokenizer" / "tokenizer_config.json").is_file()
):
    raise ValueError(f"The folder at {OUTPUT} does not contain the full experiment corpus.")
print(f"Training targets: {train.total_tokens - 1:,}   Validation tokens: {validation.total_tokens:,}")
print(f"Ready to upload: {OUTPUT}")


Opening FineWeb-Edu. The first 2,048 documents go to validation.


Training tokens:   0%|          | 0.00/12.0G [00:00<?, ?tok/s]/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Training tokens:   0%|          | 32.0M/12.0G [00:48<2:25:12, 1.37Mtok/s][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (149235 > 131072). Running this sequence through the model will result in indexing errors
Training tokens:  11%|█▏        | 1.35G/12.0G [40:19<45:10:41, 65.2ktok/s] 'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/87f09149ef4734204d70ed1d046ddc9ca3f2b8f9/sample/100BT/000_00001.parquet
Retrying in 1s [Retry 1/5].
Training tokens:  88%|████████▊ | 10.5G/12.0G [2:41:04<21:59, 1.07Mtok/s]   'peer closed connection without s

Training targets: 11,957,698,560   Validation tokens: 2,272,311
Ready to upload: /Users/seungjunlee/Downloads/fineweb_edu


When the cell prints **Ready to upload**, drag the entire **fineweb_edu** folder from Finder
into **My Drive → bmr → data** in your browser. After the upload finishes, the manifest should
be at `My Drive/bmr/data/fineweb_edu/manifest.json`. Continue with **02_standard_proxy_search**
in Colab.
